# 漫畫翻譯:PaddleOCR-VL → Qwen3.8(日 → 台灣繁中)

流程:
1. **PaddleOCR-VL-1.5** 抓文字框並辨識(實測對話 116 框 0 錯)
2. **Qwen3.8-27B** 看整頁圖片 + OCR 結果,只負責翻譯(不讓它自己認字:實測它會把口語改寫成標準語)
3. 輸出 `translate.json` 和 `report.html`(整頁標號 + 原文/譯文對照),存回 Drive

**執行階段**:L4 就夠。Qwen 走 G4 session 的 vLLM tunnel(先開好 `qwen38_27b_vllm_300k.ipynb`)。

## 1. 設定

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/manga-ocr"
PDF_PATH  = f"{DRIVE_DIR}/001bj.pdf"
PAGES     = "7-26"                      # PDF 頁碼,例:"7-26" 或 "7,9,12-15"
OUT_DIR   = f"{DRIVE_DIR}/translate"

QWEN_BASE_URL = ""                      # G4 session 印出的 BASE_URL(含 /v1)
QWEN_API_KEY  = ""
QWEN_MODEL    = "qwen3.8-27b"
CONCURRENCY   = 4                       # 同時送幾頁;vLLM 會自動批次

# 固定譯名,避免每頁譯法不同(原文=譯文)
GLOSSARY = {
    "斉藤英二郎": "齋藤英二郎",
    "斉藤": "齋藤",
    "牛田": "牛田",
    "永禄大学": "永祿大學",
    "永大": "永大",
}
MIN_BOX = 12

## 2. 安裝

Paddle 會換掉 PyTorch 依賴的 NCCL/CUDA 函式庫,所以裝在獨立 venv,用子行程執行。

In [ ]:
# 主環境:不碰 torch
!pip install -q pymupdf openai

# Paddle 獨立環境(Paddle 會換掉 torch 依賴的 NCCL/CUDA 函式庫,不能裝在一起)
!pip install -q uv
!uv venv -q --seed /content/pdenv
!uv pip install -q --python /content/pdenv/bin/python paddlepaddle-gpu==3.2.1 --index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
!uv pip install -q --python /content/pdenv/bin/python "paddleocr[doc-parser]"
!/content/pdenv/bin/python -c "import paddle; print('paddle', paddle.__version__, 'CUDA:', paddle.device.is_compiled_with_cuda())"

## 3. 從 Drive 的 PDF 抽出指定頁

In [ ]:
import os, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
import pymupdf as fitz

def parse_pages(spec):
    out = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        out += list(range(int(a), int(b or a) + 1))
    return out

assert os.path.exists(PDF_PATH), f"找不到 {PDF_PATH}"
os.makedirs(f"{OUT_DIR}/pages", exist_ok=True)
doc = fitz.open(PDF_PATH)
pages = []
for n in parse_pages(PAGES):
    page = doc[n - 1]
    imgs = page.get_images(full=True)
    q = f"{OUT_DIR}/pages/p{n:03d}"
    if len(imgs) == 1:
        x = doc.extract_image(imgs[0][0]); q += "." + x["ext"]
        open(q, "wb").write(x["image"])
    else:
        q += ".png"; page.get_pixmap(dpi=150).save(q)
    pages.append(q)
print(len(pages), "頁")

## 4. OCR:PaddleOCR-VL-1.5(獨立 venv)

In [ ]:
SCRIPT = "import json, sys\nfrom paddleocr import PaddleOCRVL\n\npages, out_path, min_box = json.load(open(sys.argv[1])), sys.argv[2], int(sys.argv[3])\n\ndef walk_blocks(obj):\n    # 欄位名稱跨版本有差,所以寬鬆地找 (bbox, text, label)\n    if isinstance(obj, dict):\n        bbox = obj.get(\"block_bbox\") or obj.get(\"coordinate\") or obj.get(\"bbox\")\n        text = obj.get(\"block_content\") or obj.get(\"text\") or obj.get(\"rec_text\")\n        if bbox is not None and len(bbox) == 4 and isinstance(text, str):\n            yield [float(v) for v in bbox], text, obj.get(\"block_label\") or obj.get(\"label\") or \"\"\n        for v in obj.values():\n            yield from walk_blocks(v)\n    elif isinstance(obj, list):\n        for v in obj:\n            yield from walk_blocks(v)\n\npipe = PaddleOCRVL(pipeline_version=\"v1.5\")\nresults, raw = {}, {}\nfor p in pages:\n    blocks, seen = [], set()\n    for res in pipe.predict(p):\n        raw.setdefault(p, []).append(res.json)\n        for bbox, text, label in walk_blocks(res.json):\n            key = tuple(round(v) for v in bbox)\n            if key in seen or not text.strip():\n                continue\n            if bbox[2] - bbox[0] < min_box or bbox[3] - bbox[1] < min_box:\n                continue\n            seen.add(key)\n            blocks.append({\"bbox\": bbox, \"label\": label, \"paddle\": text.strip()})\n    results[p] = blocks\n    print(p.rsplit(\"/\", 1)[-1], len(blocks), \"boxes\", flush=True)\n\njson.dump(results, open(out_path, \"w\"), ensure_ascii=False)\njson.dump(raw, open(out_path + \".raw.json\", \"w\"), ensure_ascii=False, default=str)\n"
open('/content/paddle_run.py', 'w').write(SCRIPT)
json.dump(pages, open('/content/pages.json', 'w'))
!/content/pdenv/bin/python /content/paddle_run.py /content/pages.json /content/paddle_out.json {MIN_BOX}
ocr = json.load(open('/content/paddle_out.json'))
print('共', sum(len(v) for v in ocr.values()), '個框')

## 5. 翻譯:Qwen 看整頁圖 + OCR 結果

In [ ]:
import base64, io, re
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from openai import OpenAI

assert QWEN_BASE_URL, "先填 QWEN_BASE_URL / QWEN_API_KEY"
client = OpenAI(base_url=QWEN_BASE_URL, api_key=QWEN_API_KEY, timeout=300)

PROMPT = '''這是一頁日文漫畫,以及 OCR 抓到的文字框(編號、座標、原文)。
請把每一框翻譯成台灣繁體中文。

規則:
- OCR 原文對「對話、旁白」很可靠,照原文翻,不要改寫原文。
- 口語要譯成口語,保留角色語氣(粗魯、敬語、疲憊等)。
- 「狀聲詞、擬態詞」的 OCR 常常是錯的:看圖判斷,能確定就意譯(例:電話鈴聲→鈴鈴鈴),不確定就 zh 填空字串。
- 框裡沒有文字(例如誤判成時鐘、圖案)的,zh 填空字串。
- 固定譯名:{glossary}

只輸出 JSON 陣列,不要其他文字:
[{{"id": 編號, "kind": "對話|旁白|狀聲詞|其他", "zh": "譯文"}}]

OCR 結果:
{boxes}'''

def page_b64(path, max_side=1600):
    im = Image.open(path).convert("RGB"); im.thumbnail((max_side, max_side))
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode()

def translate(path):
    blocks = ocr.get(path, [])
    if not blocks:
        return path, []
    boxes = "\n".join(f"{i+1}. [{', '.join(str(round(v)) for v in b['bbox'])}] {b['paddle'].replace(chr(10), ' ')}" for i, b in enumerate(blocks))
    glossary = "、".join(f"{k}={v}" for k, v in GLOSSARY.items()) or "(無)"
    r = client.chat.completions.create(
        model=QWEN_MODEL, temperature=0.3,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + page_b64(path)}},
            {"type": "text", "text": PROMPT.format(glossary=glossary, boxes=boxes)}]}],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    text = r.choices[0].message.content
    m = re.search(r"\[.*\]", text, re.S)
    try:
        out = {int(x["id"]): x for x in json.loads(m.group(0))} if m else {}
    except Exception:
        out = {}
    rows = []
    for i, b in enumerate(blocks):
        t = out.get(i + 1, {})
        rows.append({"id": i + 1, "bbox": b["bbox"], "ja": b["paddle"],
                     "kind": t.get("kind", ""), "zh": t.get("zh", "" if out else "[解析失敗]")})
    return path, rows

with ThreadPoolExecutor(CONCURRENCY) as ex:
    results = dict(ex.map(translate, pages))
for p in pages:
    print(Path(p).name, len(results[p]), "框")
json.dump({Path(p).name: v for p, v in results.items()}, open(f"{OUT_DIR}/translate.json", "w"), ensure_ascii=False, indent=1)

## 6. 報告:整頁標號 + 原文/譯文對照

In [ ]:
import html
from PIL import ImageDraw, ImageFont
font = ImageFont.load_default(size=36)
sections = []
for p in pages:
    im = Image.open(p).convert("RGB"); d = ImageDraw.Draw(im)
    for r in results[p]:
        x1, y1, x2, y2 = r["bbox"]
        d.rectangle([x1, y1, x2, y2], outline=(220, 30, 30), width=4)
        d.text((x1, max(0, y1 - 40)), str(r["id"]), fill=(220, 30, 30), font=font, stroke_width=3, stroke_fill="white")
    im.thumbnail((900, 1300)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=80)
    img = base64.b64encode(buf.getvalue()).decode()
    rows = "".join(f"<tr class='{'sfx' if r['kind'] == '狀聲詞' else ''}'><td>{r['id']}</td><td>{html.escape(r['kind'])}</td>"
                   f"<td lang=ja>{html.escape(r['ja'])}</td><td>{html.escape(r['zh'])}</td></tr>" for r in results[p])
    sections.append(f"<section><h2>{Path(p).name}</h2><div class=row><img src='data:image/jpeg;base64,{img}'>"
                    f"<table><tr><th>#</th><th>類型</th><th>原文</th><th>譯文</th></tr>{rows}</table></div></section>")
page = f"""<!doctype html><meta charset=utf-8><title>漫畫翻譯</title>
<style>body{{font:15px/1.6 system-ui,'Noto Sans TC','Noto Sans JP',sans-serif;margin:16px}}
.row{{display:flex;gap:16px;align-items:flex-start;flex-wrap:wrap}}img{{max-width:min(520px,100%)}}
table{{border-collapse:collapse;flex:1;min-width:320px}}td,th{{border:1px solid #ccc;padding:4px 6px;vertical-align:top}}
.sfx{{color:#888}}h2{{margin-top:32px}}</style>{''.join(sections)}"""
open(f"{OUT_DIR}/report.html", "w").write(page)
from IPython.display import HTML, display
display(HTML(page))
print("已存:", f"{OUT_DIR}/report.html")

## 7. 關閉執行階段

結果在 Drive 的 `manga-ocr/translate/`。G4 那邊的 Qwen session 也記得關。

In [ ]:
from google.colab import runtime
runtime.unassign()